### Acquiring GO Terms for each peptide's parent protein by mapping using uniprot ID / NCBI ID and API requests



In [ ]:
#1 
import pandas as pd
import numpy as np 


#loaad all data
df_rita_pos_selection = pd.read_excel(r"RITA_and_ABT_pos_selection_screens.xlsx",sheet_name = "RITA")
#filter poly
df_rita_pos_selection = df_rita_pos_selection[~df_rita_pos_selection['tileID'].str.contains('H')]
#filter poly
df_rita_pos_selection = df_rita_pos_selection[~df_rita_pos_selection['Gene_name'].str.contains('polyprotein')]
#filter non human positive 
df_rita_pos_selection = df_rita_pos_selection[(~df_rita_pos_selection['tileID'].str.contains('P')) | (~df_rita_pos_selection['tileID'].str.contains('N'))]

uniprots_background = df_rita_pos_selection['Uniprot_id'].dropna().unique()

NCBI_background = df_rita_pos_selection['NCBI_id'].dropna().unique()



In [ ]:
import UniProtMapper
from UniProtMapper import ProtMapper

mapper_1 = ProtMapper()

print("Mapping proteins with only ncbi labels to uniprot")

mapping_ncbi_to_uniprot, mapping_ncbi_to_uniprot_failed = mapper_1.get(ids=NCBI_background, from_db='RefSeq_Protein', to_db='UniProtKB')




In [ ]:
#establish function to query IDS;
import requests, sys
import json
def get_GOID(UPID):
    """IN: UPID, Uniprot ID
       OUT: GO IDS (list) """
    requestURL = f"https://www.ebi.ac.uk/QuickGO/services/annotation/search?geneProductId={UPID}"

    # Set header to get JSON format
    r = requests.get(requestURL, headers={ "Accept" : "application/json"})

    responsebody = r.json()

    list_goterms = []
    # Extract GO Terms
    if r.status_code == 200:
        for result in responsebody['results']:
            #print(result['goId'])
            list_goterms.append(result['goId'])
        return list_goterms 
    else:
        return None

#establish function to get names...

def get_GOTERM(GOID):
    """
    IN: (String) Go "ID" in form [GOXXXXXX]
    OUT: (String) Go "name" in form "{function}"
    """
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{GOID}"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        # The 'name' field is located within the results list
        return data['results'][0]['name']
    else:
        return None
    




def get_UPID_reviewstatus(uniprot_id):
    # UniProtKB search endpoint
    url = "https://rest.uniprot.org/uniprotkb/search"
    
    # Define parameters: search for specific ID and request 'reviewed' column
    params = {
        "query": f"accession:{uniprot_id}",
        "fields": "accession,reviewed",
        "format": "json"
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data['results']:
            # The 'entryType' field often contains the status (e.g., 'UniProtKB reviewed (Swiss-Prot)')
            status = data['results'][0].get('entryType', 'Unknown')
            if 'unreviewed' in status:
                return False 
            else:
                return True
    return pd.NA

In [ ]:
print(get_UPID_reviewstatus("A0A8B1KIC0"))

In [ ]:
#we now need to gather our total list of uniprot and NCBI IDs...
#! 
functional_df = pd.DataFrame(columns=['Uniprot ID', 'NCBI ID', 'Reviewed', 'GO IDs', 'GO Terms'])


mapped_upids = mapping_ncbi_to_uniprot['Entry']
input_NCBIds = mapping_ncbi_to_uniprot['From']

#add already-known UPIDs

all_UPIDs = list(mapped_upids) + list(uniprots_background)

functional_df['Uniprot ID'] = all_UPIDs
functional_df['NCBI ID'] = input_NCBIds

functional_df['Reviewed'] = functional_df['Uniprot ID'].apply(lambda x: get_UPID_reviewstatus(x))



In [ ]:
GO_ID_COL = []
GO_NAMES_COL = []
for UPID in functional_df['Uniprot ID']:
    #obtain GO ids
    GO_IDS = get_GOID(UPID)

    GO_names = []
    #now we grab each associated name...
    for id in GO_IDS:
        GO_names.append(get_GOTERM(id))
    GO_ID_COL.append(GO_IDS)
    GO_NAMES_COL.append(GO_names)






In [ ]:
functional_df['GO IDs'] = GO_ID_COL
functional_df['GO Terms'] = GO_NAMES_COL
functional_df.to_csv(r"GO_Term_Mapped_Viral_Proteins.csv", header=True, index=False)